In [1]:
# This file introduces an improved implementation of Variant1 of Algorithm 13,
# which is faster than the old version.
# To reproduce the results, please use the G4 GPU of google colab,
# which has 48 CPU cores and 176.88 GB of RAM.
# The improvement is partly accomplished by ChatGPT.


In [2]:
!apt-get update
!apt-get install -y libtbb-dev

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:13 http://security.ubuntu.com/ubuntu j

In [3]:
%%writefile Variant1_Algorithm_13_APPD_accelerated_by_parallel_computing.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <cassert>
#include <set>
#include <queue>
#include <random>
#include <numeric>
#include <stack>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>
#include <functional>
#include <list>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;
const double INF = numeric_limits<double>::infinity();

// Prim's MST with min-heap optimization
vector<int> primMST(const Matrix& dist) {
    int V = dist.size();
    vector<double> key(V, INF);
    vector<int> parent(V, -1);
    vector<bool> inMST(V, false);

    key[0] = 0.0;
    priority_queue<pair<double, int>, vector<pair<double, int>>, greater<>> pq;
    pq.emplace(0.0, 0);

    while (!pq.empty()) {
        auto [k, u] = pq.top(); pq.pop();
        if (inMST[u]) continue;
        inMST[u] = true;

        for (int v = 0; v < V; ++v) {
            if (dist[u][v] && !inMST[v] && dist[u][v] < key[v]) {
                key[v] = dist[u][v];
                parent[v] = u;
                pq.emplace(key[v], v);
            }
        }
    }
    return parent;
}

vector<set<int>> buildMSTGraph(int n, const vector<int>& parent) {
    vector<set<int>> mst(n);
    for (int i = 1; i < n; ++i) {
        mst[i].insert(parent[i]);
        mst[parent[i]].insert(i);
    }
    return mst;
}

void dfs(int start, const vector<set<int>>& graph, vector<bool>& visited, vector<int>& nodes) {
    stack<int> s;
    s.push(start);
    visited[start] = true;

    while (!s.empty()) {
        int node = s.top(); s.pop();
        nodes.push_back(node);
        for (int neighbor : graph[node]) {
            if (!visited[neighbor]) {
                visited[neighbor] = true;
                s.push(neighbor);
            }
        }
    }
}


void main_thread_func(
            vector<set<int>>& base_mst,
            const vector<pair<int, int>>& edge_nodes_large_to_small,
            const vector<double>& edge_weights,
            Matrix& mmj_matrix,
            deque<int>& task_queue,
            mutex& queue_mutex) {

    // auto MST_temp = base_mst;
    int current_removed = -1;
    int n = mmj_matrix.size();
    vector<bool> visited(n, false);

    vector<int> tree1, tree2;
    int task = -1;

    while (true) {

        {
            lock_guard<mutex> lock(queue_mutex);
            // if (task_queue.empty()) return;
            if (task_queue.empty()){

                auto now = chrono::system_clock::now();

                time_t now_time = chrono::system_clock::to_time_t(now);

                // cout  << "main: Current time: " << put_time(localtime(&now_time), "%Y-%m-%d %H:%M:%S") << endl;

                return;
            }

            task = task_queue.front();
            task_queue.pop_front();

        }

        // this_thread::sleep_for(chrono::milliseconds(500));

        for (int i = current_removed + 1; i <= task; ++i) {
            auto [u, v] = edge_nodes_large_to_small[i];
            base_mst[u].erase(v);
            base_mst[v].erase(u);
        }
        current_removed = task;

        auto [u, v] = edge_nodes_large_to_small[task];
        double weight = edge_weights[task];

        tree1.clear(); tree2.clear();
        dfs(u, base_mst, visited, tree1);
        dfs(v, base_mst, visited, tree2);
        fill(visited.begin(), visited.end(), false);

        for (int a : tree1)
            for (int b : tree2)
                mmj_matrix[a][b] = mmj_matrix[b][a] = weight;

    }
}

void worker(int tid,
            const vector<pair<int, int>>& edge_nodes_large_to_small,
            const vector<double>& edge_weights,
            Matrix& mmj_matrix,
            deque<int>& task_queue,
            mutex& queue_mutex) {


    int n = mmj_matrix.size();
    int num_edges = n - 1;
    int current_added = num_edges;

    vector<set<int>> MST_temp(n);

    vector<bool> visited(n, false);

    vector<int> tree1, tree2;
    int task =  - 1;

    while (true) {

        {
            lock_guard<mutex> lock(queue_mutex);
            if (task_queue.empty()){

                auto now = chrono::system_clock::now();

                time_t now_time = chrono::system_clock::to_time_t(now);

                // cout << tid << ": Current time: " << put_time(localtime(&now_time), "%Y-%m-%d %H:%M:%S") << endl;

                return;
            }
            task = task_queue.back();
            task_queue.pop_back();

        }

        if (task < num_edges - 1){
        for (int i = current_added - 1; i >= task + 1; --i) {
            auto [u, v] = edge_nodes_large_to_small[i];
            MST_temp[u].insert(v);
            MST_temp[v].insert(u);
        }

        }
        current_added = task + 1;

        auto [u, v] = edge_nodes_large_to_small[task];
        double weight = edge_weights[task];

        tree1.clear(); tree2.clear();
        dfs(u, MST_temp, visited, tree1);
        dfs(v, MST_temp, visited, tree2);

        fill(visited.begin(), visited.end(), false);

        for (int a : tree1)
            for (int b : tree2)
                mmj_matrix[a][b] = mmj_matrix[b][a] = weight;


    }
}

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(const Matrix& distance_matrix, int n_jobs) {
    int n = distance_matrix.size();
    Matrix mmj_matrix(n, vector<double>(n, 0.0));
    deque<int> task_queue;
    mutex queue_mutex;


    for (int i = 0; i < n - 1; ++i)
        task_queue.push_back(i);


    auto parent = primMST(distance_matrix);
    auto base_mst = buildMSTGraph(n, parent);

    vector<Edge> edge_list;
    for (int i = 1; i < n; ++i)
        edge_list.emplace_back(min(i, parent[i]), max(i, parent[i]), distance_matrix[i][parent[i]]);

    sort(edge_list.begin(), edge_list.end(),
         [](const Edge& a, const Edge& b) { return get<2>(a) > get<2>(b); });

    vector<pair<int, int>> edge_nodes;
    vector<double> edge_weights;
    for (const auto& [u, v, w] : edge_list) {
        edge_nodes.emplace_back(u, v);
        edge_weights.push_back(w);
    }


    vector<thread> threads;

    // use "for (int t = 0; t < n_jobs - 1; ++t)"?
    for (int t = 0; t < n_jobs; ++t)
        threads.emplace_back(worker, t,
                                cref(edge_nodes),
                                cref(edge_weights),
                                ref(mmj_matrix),
                                ref(task_queue),
                                ref(queue_mutex));


    // This part tests how long it takes to copy a MST.
    // auto start4 = chrono::high_resolution_clock::now();
    // auto MST_ttt = base_mst;
    // auto end4 = chrono::high_resolution_clock::now();
    // cout << "Time used copy mst: " << chrono::duration<double>(end4 - start4).count() << " seconds\n";


    auto start2 = chrono::high_resolution_clock::now();


    main_thread_func(base_mst,
            edge_nodes, edge_weights,
            mmj_matrix,
            task_queue,
            queue_mutex);

    auto end2 = chrono::high_resolution_clock::now();

    // cout << "Time used processing remaining: " << chrono::duration<double>(end2 - start2).count() << " seconds\n";

    auto start3 = chrono::high_resolution_clock::now();

    for (auto& th : threads) th.join();

    auto end3 = chrono::high_resolution_clock::now();

    // cout << "Time used waiting threads finish: " << chrono::duration<double>(end3 - start3).count() << " seconds\n";



    return mmj_matrix;
}



vector<vector<double>> createDistanceMatrix(int N, int seed) {
    mt19937 gen(seed);

    // Generate doubles between 1.00 and 19999.00
    uniform_real_distribution<double> dist(1.0, 19999.0);

    vector<vector<double>> A(N, vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {

            // Round to 2 decimal places
            double val = round(dist(gen) * 100.0) / 100.0;

            A[i][j] = A[j][i] = val;
        }
    }

    return A;
}

int main() {
    int N = 80221;
    int n_jobs = thread::hardware_concurrency();
    int random_seed = 78375;


    cout << "Number of nodes: "<< N << endl;
    cout << "Number of CPU cores: " << n_jobs << endl;


    auto distanceMatrix = createDistanceMatrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrix = cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(distanceMatrix, n_jobs);
    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();


    cout << fixed << setprecision(3);
    cout << "Time used for MMJ matrix (Variant1 of Algorithm 13): " << time_used << " seconds\n";
    cout << "Print last 30 values of the first row of mmj matrix: " << endl;
    const auto& row = mmjMatrix[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(2) << row[i] << " ";
    cout << "\n";

    return 0;
}



Writing Variant1_Algorithm_13_APPD_accelerated_by_parallel_computing.cpp


In [4]:
# %%writefile Variant1_Algorithm_13_APPD_accelerated_by_parallel_computing.cpp



In [5]:
%%writefile Variant1_new_implementation.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <stack>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

// ============================================================
// GRAPH
// ============================================================

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double,int>,
        vector<pair<double,int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<int>& visited,
    int token,
    vector<int>& nodes)
{
    nodes.clear();

    stack<int> st;

    st.push(start);

    visited[start] = token;

    while (!st.empty()) {

        int u = st.top();
        st.pop();

        nodes.push_back(u);

        for (const auto& e : graph[u]) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                st.push(v);
            }
        }
    }
}

// ============================================================
// MAIN THREAD
// ============================================================

void main_thread_func(
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // initially all active
    vector<char> active(num_edges, 1);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop_front();
        }

        // remove edge task
        active[task] = 0;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    vector<char> active(num_edges);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    int current_added = num_edges;

    int task;

    fill(active.begin(), active.end(), 0);

    while (true) {

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.back();
            task_queue.pop_back();
        }

        if (task < num_edges - 1){
        for (int i = current_added - 1; i >= task + 1; --i) {
            active[i] = 1;
        }
        }
        current_added = task + 1;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{


    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    // ========================================================
    // TASK QUEUE
    // ========================================================

    deque<int> task_queue;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push_back(i);

    mutex queue_mutex;

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        edge_list.emplace_back(
            min(i, parent[i]),
            max(i, parent[i]),
            distance_matrix[i][parent[i]]);
    }

    sort(
        edge_list.begin(),
        edge_list.end(),
        [](const Edge& a, const Edge& b) {

            return get<2>(a) > get<2>(b);
        });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int,int>> edge_nodes;

    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);

        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex));
    }

    // ========================================================
    // TIMING
    // ========================================================



    main_thread_func(
        graph,
        edge_nodes,
        edge_weights,
        mmj_matrix,
        task_queue,
        queue_mutex);

    for (auto& th : threads)
        th.join();



    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

vector<vector<double>> createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    vector<vector<double>> A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}


// ============================================================
// MAIN
// ============================================================

int main() {

    int N = 80221;

    int n_jobs = thread::hardware_concurrency();

    int random_seed = 7875;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix =
        createDistanceMatrix(
            N,
            random_seed);

     auto start =
        chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs);

   auto end =
        chrono::high_resolution_clock::now();

    double time_used =
        chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);

    cout << "Time used for MMJ matrix (Variant1 of Algorithm 13) - new: "
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of the first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30;
         i < row.size();
         ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}

Writing Variant1_new_implementation.cpp


In [6]:
%%writefile Reviewer_t5C9_code_cpp_version.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <limits>
#include <random>
#include <chrono>
#include <algorithm>
#include <list>
#include <iomanip>



using namespace std;

struct Edge {
    int u, v;
    double weight;
};

// Prim's MST (dense graph version)
vector<Edge> prim_mst(const vector<vector<double>>& D) {
    int n = D.size();
    vector<bool> in_mst(n, false);
    vector<double> key(n, numeric_limits<double>::infinity());
    vector<int> parent(n, -1);
    key[0] = 0.0;

    for (int count = 0; count < n; ++count) {
        double min_val = numeric_limits<double>::infinity();
        int u = -1;
        for (int i = 0; i < n; ++i)
            if (!in_mst[i] && key[i] < min_val)
                min_val = key[i], u = i;

        if (u == -1) break;

        in_mst[u] = true;
        for (int v = 0; v < n; ++v)
            if (!in_mst[v] && D[u][v] < key[v])
                key[v] = D[u][v], parent[v] = u;
    }

    vector<Edge> edges;
    for (int v = 1; v < n; ++v)
        edges.push_back({parent[v], v, D[parent[v]][v]});
    return edges;
}

// Build CSR-like structure
void build_csr(int n, const vector<Edge>& mst_edges,
               vector<int>& ptr, vector<int>& adj_edges, vector<double>& adj_weights) {
    vector<int> edge_counts(n, 0);
    for (const auto& e : mst_edges) {
        edge_counts[e.u]++;
        edge_counts[e.v]++;
    }

    ptr.resize(n + 1);
    for (int i = 1; i <= n; ++i)
        ptr[i] = ptr[i - 1] + edge_counts[i - 1];

    adj_edges.resize(ptr[n]);
    adj_weights.resize(ptr[n]);
    vector<int> positions(n, 0);

    for (const auto& e : mst_edges) {
        for (int i = 0; i < 2; ++i) {
            int u = (i == 0) ? e.u : e.v;
            int v = (i == 0) ? e.v : e.u;
            int idx = ptr[u] + positions[u]++;
            adj_edges[idx] = v;
            adj_weights[idx] = e.weight;
        }
    }
}


#include <tbb/parallel_for.h>
#include <tbb/blocked_range.h>
#include <tbb/parallel_for_each.h>

vector<vector<double>> compute_bottleneck_matrix(int n,
    const vector<int>& ptr, const vector<int>& adj_edges, const vector<double>& adj_weights) {

    vector<vector<double>> bottleneck(n, vector<double>(n, 0.0));

    tbb::parallel_for(tbb::blocked_range<int>(0, n),
        [&](const tbb::blocked_range<int>& r) {
            for (int src = r.begin(); src < r.end(); ++src) {
                vector<bool> visited(n, false);
                vector<double> max_edges(n, 0.0);
                queue<pair<int, double>> q;

                visited[src] = true;
                q.push({src, 0.0});

                while (!q.empty()) {
                    auto [u, curr_max] = q.front(); q.pop();

                    for (int i = ptr[u]; i < ptr[u + 1]; ++i) {
                        int v = adj_edges[i];
                        double weight = adj_weights[i];

                        if (!visited[v]) {
                            double new_max = max(curr_max, weight);
                            visited[v] = true;
                            max_edges[v] = new_max;
                            q.push({v, new_max});
                        }
                    }
                }

                bottleneck[src] = move(max_edges); // safe: each src owns its row
            }
        }
    );

    return bottleneck;
}


vector<vector<double>> ultra_fast_wide(const vector<vector<double>>& D) {
    int n = D.size();


    vector<Edge> mst = prim_mst(D);

    vector<int> ptr, adj_edges;
    vector<double> adj_weights;
    build_csr(n, mst, ptr, adj_edges, adj_weights);

    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights);
}



vector<vector<double>> createDistanceMatrix(int N, int seed) {
    mt19937 gen(seed);

    // Generate doubles between 1.00 and 19999.00
    uniform_real_distribution<double> dist(1.0, 19999.0);

    vector<vector<double>> A(N, vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {

            // Round to 2 decimal places
            double val = round(dist(gen) * 100.0) / 100.0;

            A[i][j] = A[j][i] = val;
        }
    }

    return A;
}

int main() {


    int N = 80221;
    int n_jobs = thread::hardware_concurrency();
    int random_seed = 78375;


    cout << "Number of nodes: "<< N << endl;
    cout << "Number of CPU cores: " << n_jobs << endl;



    auto distanceMatrix = createDistanceMatrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrix = ultra_fast_wide(distanceMatrix);
    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);
    cout << "Time used for MMJ matrix (Reviewer t5C9's code cpp version): " << time_used << " seconds\n";
    cout << "Print last 30 values of the first row of mmj matrix: " << endl;
    const auto& row = mmjMatrix[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(2) << row[i] << " ";
    cout << "\n";

    return 0;

}



Writing Reviewer_t5C9_code_cpp_version.cpp


In [7]:
!g++ -std=c++17 -O3 -march=native Variant1_Algorithm_13_APPD_accelerated_by_parallel_computing.cpp  -o tt -ltbb
!./tt

Number of nodes: 80221
Number of CPU cores: 48
Time used for MMJ matrix (Variant1 of Algorithm 13): 30.215 seconds
Print last 30 values of the first row of mmj matrix: 
1.71 1.71 1.71 1.71 2.76 1.71 1.71 1.83 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 


In [8]:
!g++ -std=c++17 -O3 -march=native Variant1_new_implementation.cpp  -o tt -ltbb
!./tt

Number of nodes: 80221
Number of CPU cores: 48
Time used for MMJ matrix (Variant1 of Algorithm 13) - new: 24.664 seconds
Print last 30 values of the first row of mmj matrix:
1.46 1.46 1.53 1.48 1.55 1.42 1.26 1.46 1.26 1.42 1.93 1.76 1.27 1.26 1.27 1.61 1.44 1.34 2.04 1.37 1.28 1.27 1.31 1.26 1.27 1.39 1.28 1.46 1.29 1.40 


In [9]:
!g++ -std=c++17 -O3 -march=native Reviewer_t5C9_code_cpp_version.cpp  -o tt -ltbb
!./tt

Number of nodes: 80221
Number of CPU cores: 48
Time used for MMJ matrix (Reviewer t5C9's code cpp version): 39.168 seconds
Print last 30 values of the first row of mmj matrix: 
1.71 1.71 1.71 1.71 2.76 1.71 1.71 1.83 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 1.71 


In [10]:
import platform
import psutil

# CPU information
print("CPU Information:")
print(f"Processor: {platform.processor()}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU frequency: {psutil.cpu_freq().current:.2f} MHz")

# RAM information
ram = psutil.virtual_memory()

print("\nRAM Information:")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

CPU Information:
Processor: x86_64
Physical cores: 24
Logical cores: 48
CPU frequency: 3505.10 MHz

RAM Information:
Total RAM: 176.88 GB
